In [1]:
"""
Q4.1  单参数敏感性扫描
  对 c_buy, c_m, c_l 各自在 ±30% 范围扫描 (-30, -20, -10, 0, +10, +20, +30)%
  每个场景下用 Q3 已存的 L/n_M/n_L 重新计算 EAC, 找最优 (T_M*, T_L*)

加速思路：寿命 L、维护次数 n_M, n_L 只依赖 (i, T_M, T_L)，与成本无关。
所以只需读 Q3 的 eac_grid_all.csv，重算 EAC 即可，无需重跑仿真。

输出:
  tables/sensitivity_single_param.csv   每场景每台的最优结果
  tables/sensitivity_total_fleet.csv    每场景全队 EAC 总和
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT  # 修正：指向项目根目录

grid = pd.read_csv(ROOT / "Q3输出/tables/eac_grid_all.csv")
print(f"Loaded {len(grid)} grid rows from Q3")

C_BUY = 300.0
C_M = 3.0
C_L = 12.0

levels = np.array([-30, -20, -10, 0, 10, 20, 30]) / 100  # 7

def best_per_filter(c_buy, c_m, c_l):
    g = grid.copy()
    g["EAC_new"] = (c_buy + g["n_M"]*c_m + g["n_L"]*c_l) / g["L_years"]
    rows = []
    for i in range(1, 11):
        sub = g[g.i == i]
        b = sub.loc[sub["EAC_new"].idxmin()]
        T_L_disp = "inf" if b["T_L_label"] == np.inf or b["T_L"] == 99999 else f"{int(b['T_L_label'])}"
        rows.append(dict(
            i=i,
            T_M=int(b["T_M"]),
            T_L=T_L_disp,
            L_years=round(b["L_years"], 2),
            retired=bool(b["retired"]),
            EAC=round(b["EAC_new"], 3),
        ))
    return pd.DataFrame(rows)

# 三个参数分别扫描
all_records = []
fleet_records = []

for param_name, c_base in [("c_buy", C_BUY), ("c_m", C_M), ("c_l", C_L)]:
    for level in levels:
        c_val = c_base * (1 + level)
        if param_name == "c_buy":
            df = best_per_filter(c_val, C_M, C_L)
        elif param_name == "c_m":
            df = best_per_filter(C_BUY, c_val, C_L)
        else:  # c_l
            df = best_per_filter(C_BUY, C_M, c_val)
        df["param"] = param_name
        df["level"] = level
        df["c_value"] = round(c_val, 3)
        all_records.append(df)
        fleet_records.append(dict(
            param=param_name,
            level=level,
            c_value=round(c_val, 3),
            EAC_total=round(df["EAC"].sum(), 2),
        ))

sweep = pd.concat(all_records, ignore_index=True)
fleet = pd.DataFrame(fleet_records)

sweep.to_csv(ROOT / "Q4输出/tables/sensitivity_single_param.csv", index=False)
fleet.to_csv(ROOT / "Q4输出/tables/sensitivity_total_fleet.csv", index=False)

print(f"\nSaved:")
print(f"  sensitivity_single_param.csv  ({len(sweep)} rows)")
print(f"  sensitivity_total_fleet.csv   ({len(fleet)} rows)")

# 摘要打印：全队 EAC 在各场景下的变化
print("\n=== 全队 EAC 总和（万元/年）===")
piv = fleet.pivot_table(index="param", columns="level", values="EAC_total")
piv.columns = [f"{int(l*100):+d}%" for l in piv.columns]
print(piv.round(2).to_string())

# 弹性 (中心差分)
def elasticity(series_levels, series_eac, base_idx):
    """围绕 base 的弹性 ε = (dEAC/EAC) / (dc/c)"""
    e_plus = series_eac.iloc[base_idx + 1] - series_eac.iloc[base_idx]
    e_minus = series_eac.iloc[base_idx] - series_eac.iloc[base_idx - 1]
    dEAC = (e_plus + e_minus) / 2
    dC_pct = 0.10  # ±10% 步长
    return (dEAC / series_eac.iloc[base_idx]) / dC_pct

base_idx = 3  # level=0
print("\n=== 全队 EAC 弹性 (围绕基准点) ===")
for p in ["c_buy", "c_m", "c_l"]:
    sub = fleet[fleet.param == p].sort_values("level").reset_index(drop=True)
    eps = elasticity(sub["level"], sub["EAC_total"], base_idx)
    print(f"  {p}: ε = {eps:.4f}  "
          f"(EAC at -30%/0/+30% = {sub['EAC_total'].iloc[0]:.1f} / "
          f"{sub['EAC_total'].iloc[base_idx]:.1f} / {sub['EAC_total'].iloc[-1]:.1f})")


Loaded 810 grid rows from Q3

Saved:
  sensitivity_single_param.csv  (210 rows)
  sensitivity_total_fleet.csv   (21 rows)

=== 全队 EAC 总和（万元/年）===
          -30%     -20%     -10%      +0%     +10%     +20%     +30%
param                                                               
c_buy   928.72  1034.84  1138.55  1241.59  1344.48  1445.57  1545.58
c_l    1215.57  1225.19  1233.68  1241.59  1249.50  1257.21  1263.30
c_m    1197.95  1213.94  1228.30  1241.59  1254.80  1267.82  1280.38

=== 全队 EAC 弹性 (围绕基准点) ===
  c_buy: ε = 0.8293  (EAC at -30%/0/+30% = 928.7 / 1241.6 / 1545.6)
  c_m: ε = 0.1067  (EAC at -30%/0/+30% = 1198.0 / 1241.6 / 1280.4)
  c_l: ε = 0.0637  (EAC at -30%/0/+30% = 1215.6 / 1241.6 / 1263.3)


In [2]:
"""
Q4.2  弹性分析与最优方案切换检查
  - 每台 EAC* 关于 c_buy / c_m / c_l 的弹性（围绕基准的中心差分）
  - 标记最优 (T_M*, T_L*) 在哪些场景下切换
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT  # 修正：指向项目根目录

sweep = pd.read_csv(ROOT / "Q4输出/tables/sensitivity_single_param.csv")

# ===== 切换检查 =====
print("=== 最优 (T_M*, T_L*) 切换 ===")
switching = []
for i in range(1, 11):
    sub = sweep[sweep.i == i]
    base = sub[(sub.param == "c_buy") & (sub.level == 0)].iloc[0]
    base_rule = (base["T_M"], base["T_L"])
    print(f"\nA{i}  基准最优: T_M={base['T_M']}, T_L={base['T_L']}, EAC={base['EAC']:.2f}")
    for p in ["c_buy", "c_m", "c_l"]:
        psub = sub[sub.param == p].sort_values("level")
        unique_rules = psub[["level","T_M","T_L"]].drop_duplicates(subset=["T_M","T_L"])
        if len(unique_rules) > 1:
            for _, r in unique_rules.iterrows():
                rule = (r["T_M"], r["T_L"])
                marker = " ← BASE" if rule == base_rule else ""
                print(f"    {p:5s} level={int(r['level']*100):+3d}%: T_M={r['T_M']}, T_L={r['T_L']}{marker}")
                switching.append(dict(
                    i=i, param=p, level=r["level"],
                    T_M=r["T_M"], T_L=r["T_L"],
                    is_base=rule==base_rule,
                ))

sw_df = pd.DataFrame(switching)
sw_df.to_csv(ROOT / "Q4输出/tables/optimal_rule_switching.csv", index=False)
print(f"\n切换记录已保存到 optimal_rule_switching.csv ({len(sw_df)} 条)")

# ===== 每台弹性 =====
print("\n=== 每台 EAC* 弹性（围绕基准）===")
elast_rows = []
for i in range(1, 11):
    sub = sweep[sweep.i == i]
    base_eac = sub[(sub.param == "c_buy") & (sub.level == 0)].iloc[0]["EAC"]
    rec = dict(i=i, EAC_base=round(base_eac, 2))
    for p in ["c_buy", "c_m", "c_l"]:
        psub = sub[sub.param == p].sort_values("level").reset_index(drop=True)
        # central diff at level=0 (idx=3)
        e_plus = psub.iloc[4]["EAC"] - psub.iloc[3]["EAC"]
        e_minus = psub.iloc[3]["EAC"] - psub.iloc[2]["EAC"]
        dEAC = (e_plus + e_minus) / 2
        eps = (dEAC / base_eac) / 0.10
        rec[f"eps_{p}"] = round(eps, 4)
    elast_rows.append(rec)
elast = pd.DataFrame(elast_rows)
print(elast.to_string(index=False))
elast.to_csv(ROOT / "Q4输出/tables/per_filter_elasticity.csv", index=False)

# ===== 每台 ±30% 极端场景 =====
print("\n=== 每台 ±30% 极端 EAC ===")
for i in range(1, 11):
    sub = sweep[sweep.i == i]
    print(f"\nA{i}:")
    for p in ["c_buy", "c_m", "c_l"]:
        psub = sub[sub.param == p].sort_values("level")
        eac_min = psub["EAC"].iloc[0]   # -30%
        eac_base = psub[psub.level==0]["EAC"].iloc[0]
        eac_max = psub["EAC"].iloc[-1]  # +30%
        print(f"  {p:5s}: -30% → {eac_min:.1f},  base → {eac_base:.1f},  +30% → {eac_max:.1f}  "
              f"(range = {eac_max - eac_min:.1f})")


=== 最优 (T_M*, T_L*) 切换 ===

A1  基准最优: T_M=80, T_L=450.0, EAC=94.29
    c_buy level=-30%: T_M=120.0, T_L=540.0
    c_buy level=-10%: T_M=80.0, T_L=450.0 ← BASE
    c_buy level=+30%: T_M=60.0, T_L=450.0
    c_m   level=-30%: T_M=60.0, T_L=450.0
    c_m   level=-20%: T_M=80.0, T_L=450.0 ← BASE
    c_m   level=+20%: T_M=120.0, T_L=540.0

A2  基准最优: T_M=100, T_L=540.0, EAC=45.00

A3  基准最优: T_M=90, T_L=540.0, EAC=99.64
    c_m   level=-30%: T_M=80.0, T_L=540.0
    c_m   level=-20%: T_M=90.0, T_L=540.0 ← BASE

A4  基准最优: T_M=120, T_L=inf, EAC=34.13

A5  基准最优: T_M=60, T_L=720.0, EAC=195.60
    c_buy level=-30%: T_M=80.0, T_L=720.0
    c_buy level=-20%: T_M=70.0, T_L=720.0
    c_buy level=-10%: T_M=60.0, T_L=720.0 ← BASE
    c_buy level=+30%: T_M=50.0, T_L=240.0
    c_m   level=-30%: T_M=50.0, T_L=720.0
    c_m   level=-20%: T_M=60.0, T_L=720.0 ← BASE
    c_m   level=+30%: T_M=80.0, T_L=720.0
    c_l   level=-30%: T_M=60.0, T_L=240.0
    c_l   level=-20%: T_M=60.0, T_L=720.0 ← BASE

A6  基准最优: T_M

In [3]:
"""
Q4.3  双参数联合敏感性
  c_buy × c_l 联合扫描（c_m 占比小，固定为 base）
  - 网格 7×7 = 49 场景
  - 全队 EAC 总和热力图
  - 标记最坏 / 最好场景
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT  # 修正：指向项目根目录

grid = pd.read_csv(ROOT / "Q3输出/tables/eac_grid_all.csv")

C_BUY_BASE = 300.0
C_M_BASE = 3.0
C_L_BASE = 12.0

levels = np.array([-30, -20, -10, 0, 10, 20, 30]) / 100  # 7 levels

def total_optimal_eac(c_buy, c_m, c_l):
    g = grid.copy()
    g["EAC_new"] = (c_buy + g["n_M"]*c_m + g["n_L"]*c_l) / g["L_years"]
    total = 0.0
    for i in range(1, 11):
        sub = g[g.i == i]
        total += sub["EAC_new"].min()
    return total

# c_buy × c_l (c_m fixed at base)
records = []
for lb in levels:
    cb = C_BUY_BASE * (1 + lb)
    for ll in levels:
        cl = C_L_BASE * (1 + ll)
        eac = total_optimal_eac(cb, C_M_BASE, cl)
        records.append(dict(
            level_buy=lb, level_l=ll,
            c_buy=cb, c_l=cl,
            EAC_total=round(eac, 2),
        ))
joint_buy_l = pd.DataFrame(records)
joint_buy_l.to_csv(ROOT / "Q4输出/tables/joint_buy_l.csv", index=False)

print("=== c_buy × c_l 联合 (c_m=3 固定) — 全队 EAC 总和 ===")
piv = joint_buy_l.pivot_table(index="level_buy", columns="level_l", values="EAC_total")
piv.index = [f"c_buy {int(l*100):+d}%" for l in piv.index]
piv.columns = [f"c_l {int(l*100):+d}%" for l in piv.columns]
print(piv.round(1).to_string())

best = joint_buy_l.loc[joint_buy_l["EAC_total"].idxmin()]
worst = joint_buy_l.loc[joint_buy_l["EAC_total"].idxmax()]
print(f"\n最好场景: c_buy {int(best['level_buy']*100):+d}%, c_l {int(best['level_l']*100):+d}% "
      f"→ EAC = {best['EAC_total']:.1f}")
print(f"最坏场景: c_buy {int(worst['level_buy']*100):+d}%, c_l {int(worst['level_l']*100):+d}% "
      f"→ EAC = {worst['EAC_total']:.1f}")
print(f"极差 = {worst['EAC_total'] - best['EAC_total']:.1f} 万元/年")

# c_buy × c_m
records = []
for lb in levels:
    cb = C_BUY_BASE * (1 + lb)
    for lm in levels:
        cm = C_M_BASE * (1 + lm)
        eac = total_optimal_eac(cb, cm, C_L_BASE)
        records.append(dict(
            level_buy=lb, level_m=lm,
            c_buy=cb, c_m=cm,
            EAC_total=round(eac, 2),
        ))
joint_buy_m = pd.DataFrame(records)
joint_buy_m.to_csv(ROOT / "Q4输出/tables/joint_buy_m.csv", index=False)
print(f"\nSaved: joint_buy_l.csv, joint_buy_m.csv")

# 三参数同时扰动 ±30% 的极端
print("\n=== 三参数同向扰动 ===")
for sign in [-1, +1]:
    s = sign * 0.30
    eac = total_optimal_eac(C_BUY_BASE*(1+s), C_M_BASE*(1+s), C_L_BASE*(1+s))
    print(f"  全部 {int(s*100):+d}%: EAC 总和 = {eac:.2f} 万元/年")
print(f"  基准:        EAC 总和 = {total_optimal_eac(C_BUY_BASE, C_M_BASE, C_L_BASE):.2f} 万元/年")


=== c_buy × c_l 联合 (c_m=3 固定) — 全队 EAC 总和 ===
            c_l -30%  c_l -20%  c_l -10%  c_l +0%  c_l +10%  c_l +20%  c_l +30%
c_buy -30%     907.3     915.1     922.8    928.7     934.5     940.2     945.9
c_buy -20%    1011.6    1019.4    1027.1   1034.8    1041.0    1046.9    1052.8
c_buy -10%    1114.2    1122.7    1130.6   1138.6    1146.3    1152.3    1158.2
c_buy +0%     1215.6    1225.2    1233.7   1241.6    1249.5    1257.2    1263.3
c_buy +10%    1315.9    1326.3    1335.9   1344.5    1352.5    1360.2    1367.8
c_buy +20%    1415.4    1425.8    1436.2   1445.6    1454.1    1462.6    1470.1
c_buy +30%    1512.9    1524.6    1535.2   1545.6    1554.7    1563.3    1571.9

最好场景: c_buy -30%, c_l -30% → EAC = 907.3
最坏场景: c_buy +30%, c_l +30% → EAC = 1571.9
极差 = 664.5 万元/年

Saved: joint_buy_l.csv, joint_buy_m.csv

=== 三参数同向扰动 ===
  全部 -30%: EAC 总和 = 869.12 万元/年
  全部 +30%: EAC 总和 = 1614.07 万元/年
  基准:        EAC 总和 = 1241.59 万元/年
